# Находим мошеннические операции

### Подключаем библиотеки

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
RANDOM_STATE = 42


## Загружаем данные

### Читаем датасет

In [ ]:
df = pd.read_csv("creditcard.csv")

df["Hour"] = ((df["Time"] // 3600) % 24).astype(int)
df["Day"] = ((df["Time"] // 86400) % 7).astype(int)

print(f"Shape: {df.shape}")
print(f"Duplicates: {df.duplicated().sum()}")
print("Missing values:")
print(df.isna().sum().sort_values(ascending=False).head())
print()
print("Class distribution:")
print(df["Class"].value_counts())

df.head()


## Изучаем данные

### Считаем классы

In [ ]:
class_share = df["Class"].value_counts(normalize=True).rename(index={0: "Normal", 1: "Fraud"})
print(class_share)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(df.loc[df["Class"] == 1, "Amount"], bins=80, kde=True, color="tomato", ax=axes[0, 0])
axes[0, 0].set_title("Fraud Amount Distribution")
axes[0, 0].set_xlim(0, 1000)

sns.histplot(df.loc[df["Class"] == 0, "Amount"], bins=80, kde=True, color="steelblue", ax=axes[0, 1])
axes[0, 1].set_title("Normal Amount Distribution")
axes[0, 1].set_xlim(0, 2000)

sns.countplot(data=df[df["Class"] == 1], x="Hour", color="tomato", ax=axes[1, 0])
axes[1, 0].set_title("Fraud by Hour")
axes[1, 0].set_xlabel("Hour")

sns.countplot(data=df[df["Class"] == 1], x="Day", color="darkorange", ax=axes[1, 1])
axes[1, 1].set_title("Fraud by Day")
axes[1, 1].set_xlabel("Day")

plt.tight_layout()
plt.show()


## Готовим признаки

### Собираем признаки

In [ ]:
base_features = df.drop(columns=["Class"])
y = df["Class"]

base_scaler = StandardScaler()
X_scaled = base_scaler.fit_transform(base_features)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

X_train_normal = X_train[y_train == 0]
scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")


## Сравниваем модели

### Считаем метрики

In [ ]:
def evaluate_detector(name, model, X_train_data, X_test_data, y_train_data, y_test_data, unsupervised=False):
    if unsupervised:
        model.fit(X_train_data)
        raw_scores = -model.decision_function(X_test_data)
        predictions = (model.predict(X_test_data) == -1).astype(int)
    else:
        model.fit(X_train_data, y_train_data)
        raw_scores = model.predict_proba(X_test_data)[:, 1]
        predictions = (raw_scores >= 0.5).astype(int)

    roc_auc = roc_auc_score(y_test_data, raw_scores)
    pr_auc = average_precision_score(y_test_data, raw_scores)
    precision = precision_score(y_test_data, predictions)
    recall = recall_score(y_test_data, predictions)

    return {
        "model": name,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "precision": precision,
        "recall": recall,
        "scores": raw_scores,
        "predictions": predictions,
    }

baseline_models = {
    "OneClassSVM": OneClassSVM(nu=y.mean(), kernel="rbf", gamma="scale"),
    "IsolationForest": IsolationForest(
        n_estimators=150,
        contamination=y.mean(),
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "XGBoost": XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="aucpr",
        random_state=RANDOM_STATE,
    ),
}

results = []
results.append(evaluate_detector("OneClassSVM", baseline_models["OneClassSVM"], X_train_normal, X_test, y_train, y_test, unsupervised=True))
results.append(evaluate_detector("IsolationForest", baseline_models["IsolationForest"], X_train_normal, X_test, y_train, y_test, unsupervised=True))
results.append(evaluate_detector("XGBoost", baseline_models["XGBoost"], X_train, X_test, y_train, y_test, unsupervised=False))

results_df = pd.DataFrame(results).drop(columns=["scores", "predictions"]).sort_values("pr_auc", ascending=False)
results_df


### Выбираем baseline

In [ ]:
best_baseline = max(results, key=lambda item: item["pr_auc"])
print(f"Best baseline model by PR-AUC: {best_baseline['model']}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for result in results:
    fpr, tpr, _ = roc_curve(y_test, result["scores"])
    precision, recall, _ = precision_recall_curve(y_test, result["scores"])

    axes[0].plot(fpr, tpr, label=f"{result['model']} (AUC={result['roc_auc']:.3f})")
    axes[1].plot(recall, precision, label=f"{result['model']} (AP={result['pr_auc']:.3f})")

axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[0].set_title("ROC Curves")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend()

axes[1].set_title("Precision-Recall Curves")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend()

plt.tight_layout()
plt.show()


## Добавляем признаки

### Строим признаки

In [ ]:
df_features = df.copy()
df_features["Is_deep_night"] = df_features["Hour"].isin([1, 2, 3, 4]).astype(int)
df_features["Is_morning"] = df_features["Hour"].isin([9, 10, 11]).astype(int)
df_features["Amount_log"] = np.log1p(df_features["Amount"])
df_features["Svm_score"] = -baseline_models["OneClassSVM"].decision_function(X_scaled)
df_features["Forest_score"] = -baseline_models["IsolationForest"].decision_function(X_scaled)

feature_matrix = df_features.drop(columns=["Class"])
y_enhanced = df_features["Class"]

enhanced_scaler = StandardScaler()
X_enhanced = enhanced_scaler.fit_transform(feature_matrix)

X_train_enh, X_test_enh, y_train_enh, y_test_enh = train_test_split(
    X_enhanced,
    y_enhanced,
    test_size=0.2,
    stratify=y_enhanced,
    random_state=RANDOM_STATE,
)

X_train_tune, X_val_tune, y_train_tune, y_val_tune = train_test_split(
    X_train_enh,
    y_train_enh,
    test_size=0.25,
    stratify=y_train_enh,
    random_state=RANDOM_STATE,
)

print(f"Enhanced feature count: {feature_matrix.shape[1]}")


## Настраиваем XGBoost

### Задаем цель

In [ ]:
def objective(trial):
    params = {
        "objective": "binary:logistic",
        "eval_metric": "aucpr",
        "n_estimators": trial.suggest_int("n_estimators", 200, 1200),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 5.0),
        "max_delta_step": trial.suggest_float("max_delta_step", 0.0, 10.0),
        "scale_pos_weight": y_train_enh.value_counts()[0] / y_train_enh.value_counts()[1],
        "random_state": RANDOM_STATE,
    }

    model = XGBClassifier(**params)
    model.fit(X_train_tune, y_train_tune)
    probabilities = model.predict_proba(X_val_tune)[:, 1]
    return average_precision_score(y_val_tune, probabilities)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30, show_progress_bar=False)

print("Best params:")
print(study.best_params)
print(f"Best validation PR-AUC: {study.best_value:.4f}")


### Собираем лучшую модель

In [ ]:
best_model = XGBClassifier(**study.best_params)
best_model.fit(X_train_enh, y_train_enh)

test_scores = best_model.predict_proba(X_test_enh)[:, 1]
test_predictions = (test_scores >= 0.5).astype(int)

print(classification_report(y_test_enh, test_predictions, digits=4))
print("Confusion matrix:")
print(confusion_matrix(y_test_enh, test_predictions))
print(f"ROC-AUC: {roc_auc_score(y_test_enh, test_scores):.4f}")
print(f"PR-AUC: {average_precision_score(y_test_enh, test_scores):.4f}")


## Подбираем порог

### Ищем порог

In [ ]:
def find_threshold_by_recall(y_true, probabilities, min_recall=0.83):
    precision, recall, thresholds = precision_recall_curve(y_true, probabilities)
    best_threshold = 0.5
    best_precision = 0.0

    for p, r, t in zip(precision[:-1], recall[:-1], thresholds):
        if r >= min_recall and p > best_precision:
            best_precision = p
            best_threshold = t

    return best_threshold, best_precision

best_threshold, best_precision = find_threshold_by_recall(y_test_enh, test_scores, min_recall=0.83)
threshold_predictions = (test_scores >= best_threshold).astype(int)

precision_curve, recall_curve, _ = precision_recall_curve(y_test_enh, test_scores)
plt.figure(figsize=(8, 5))
plt.plot(recall_curve, precision_curve, color="crimson")
plt.title("Precision-Recall Curve for Tuned XGBoost")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.show()

print(f"Selected threshold: {best_threshold:.4f}")
print(f"Precision at threshold: {precision_score(y_test_enh, threshold_predictions):.4f}")
print(f"Recall at threshold: {recall_score(y_test_enh, threshold_predictions):.4f}")


## Разбираем ошибки

### Проверяем ошибки

In [ ]:
analysis_df = df_features.copy()
all_scores = best_model.predict_proba(X_enhanced)[:, 1]
analysis_df["predicted_label"] = (all_scores >= best_threshold).astype(int)
analysis_df["FP"] = ((analysis_df["predicted_label"] == 1) & (analysis_df["Class"] == 0)).astype(int)
analysis_df["FN"] = ((analysis_df["predicted_label"] == 0) & (analysis_df["Class"] == 1)).astype(int)

print("False positives:", int(analysis_df["FP"].sum()))
print("False negatives:", int(analysis_df["FN"].sum()))


### Рисуем графики

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

analysis_df.loc[analysis_df["FP"] == 1].groupby("Hour").size().plot(kind="bar", ax=axes[0, 0], color="salmon")
axes[0, 0].set_title("False Positives by Hour")
axes[0, 0].set_xlabel("Hour")
axes[0, 0].set_ylabel("Count")

analysis_df.loc[analysis_df["FP"] == 1].groupby("Day").size().plot(kind="bar", ax=axes[0, 1], color="sandybrown")
axes[0, 1].set_title("False Positives by Day")
axes[0, 1].set_xlabel("Day")
axes[0, 1].set_ylabel("Count")

analysis_df.loc[analysis_df["FN"] == 1].groupby("Hour").size().plot(kind="bar", ax=axes[1, 0], color="slateblue")
axes[1, 0].set_title("False Negatives by Hour")
axes[1, 0].set_xlabel("Hour")
axes[1, 0].set_ylabel("Count")

analysis_df.loc[analysis_df["FN"] == 1].groupby("Day").size().plot(kind="bar", ax=axes[1, 1], color="mediumpurple")
axes[1, 1].set_title("False Negatives by Day")
axes[1, 1].set_xlabel("Day")
axes[1, 1].set_ylabel("Count")

plt.tight_layout()
plt.show()


## Подводим итоги